# SPIN 7B — Step 1 (AUTHOR-SAMPLE variant): importance -> coupled indices

**Only one thing changes** versus the faithful `7B testing/SPIN_7B_step1_importance.ipynb`:
the calibration-sample selection now uses the **authors' exact method** —
`datasets.Dataset.shuffle(seed=0).select(range(128))` (their `utils.get_loaders(disentangle=True)`),
instead of Python's `random.Random(0).shuffle`. Same seed, but a different generator picks a
different 128 samples, and importance is computed on *those* samples — so this can shift the coupled set.

Everything else is identical: importance metric `|W|*Σ|∇L|`, `get_set_difference_mask`, p=q=5e-7,
MLP target, grouped scoring, fp16, `MAXLEN=1024` guard, checkpointing. This isolates the
sample-selection variable so we can A/B against the current 1,149-weight result.

**Output:** `coupled_indices_authorsample.json` (separate file — does NOT overwrite `coupled_indices.json`).
Compare `coupled_count` against the faithful **1,149**.

Also pins **`datasets==2.18.0`** (authors' version) so the shuffle permutation is bit-reproducible.


## Step 0 — env + config

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys
# authors' env: datasets==2.18.0 (bit-reproducible shuffle). transformers pin unchanged for now (separate variable).
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','datasets==2.18.0','sentencepiece'], check=False)
import random, gc, time, json
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

MODEL_ID='Qwen/Qwen2-7B-Instruct'
NSAMPLES=128
TARGET='mlp'
MAXLEN=1024          # generous cap (authors don't truncate; their samples are short). Safety only.
LAYER_GROUP=4
P=5e-7; Q=5e-7       # authors' Qwen2-7B extraction ratio (p and q)
SEED=0
DTYPE=torch.float16
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
def memline(tag=''):
    s=''
    for d in range(torch.cuda.device_count()):
        f,t=torch.cuda.mem_get_info(d); s+=f' GPU{d} {f/1e9:.1f}/{t/1e9:.1f}'
    print(f'  [{tag}]{s} GiB free')


## Step 1 — data + model (AUTHOR sample selection: HF datasets shuffle+select)


In [ ]:
if not os.path.exists('DEAN'):
    subprocess.run(['git','clone','--depth','1','https://github.com/AI45Lab/DEAN.git'], check=True)
DATA='DEAN/src/data'
CSV={'g':f'{DATA}/alpaca_cleaned_no_safety_train.csv',
     'f':f'{DATA}/beaver_train330k_fairness_safe_1k.csv',
     'p':f'{DATA}/beaver_train330k_privacy_safe_1k.csv'}

tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'})
model.eval()
GEN_DEV=model.get_input_embeddings().weight.device
print('model split loaded | entry', GEN_DEV); memline('loaded')

def target_names(m): return [n for n,p in m.named_parameters() if p.dim()==2 and 'weight' in n and TARGET in n]

# ---- authors' EXACT sample selection (utils.get_loaders disentangle=True), verbatim logic ----
#   traindata.shuffle(seed=seed).select(range(nsamples));  inp = prompt + response[:,1:];
#   tar = inp with the prompt tokens masked to -100 (loss on response only).
def make_batches(csv_path, n):
    traindata = load_dataset('csv', data_files={'train':csv_path}, split='train')
    sampled   = traindata.shuffle(seed=SEED).select(range(n))   # <-- the ONLY change vs faithful Step 1
    out=[]
    for i in range(n):
        ep=tokenizer(str(sampled['prompt'][i]),   return_tensors='pt').input_ids
        er=tokenizer(str(sampled['response'][i]), return_tensors='pt').input_ids[:,1:]  # authors' unconditional BOS drop
        inp=torch.cat([ep,er],1)[:,:MAXLEN].to(GEN_DEV); tar=inp.clone(); tar[:,:ep.shape[1]]=-100
        if (tar!=-100).sum()>0: out.append((inp,tar))
    return out
BATCHES={k:make_batches(CSV[k],NSAMPLES) for k in CSV}
print('batches | g',len(BATCHES['g']),'| f',len(BATCHES['f']),'| p',len(BATCHES['p']),'(author shuffle+select)')


## Step 2 — score group-by-group, derive coupled indices, save only those

In [ ]:
names=target_names(model); nL=model.config.num_hidden_layers
GROUPS=[list(range(g,min(g+LAYER_GROUP,nL))) for g in range(0,nL,LAYER_GROUP)]
CK='/kaggle/working/coupled_ckpt.pt'

if os.path.exists(CK):
    st=torch.load(CK); coupled=st['coupled']; cstats=st.get('cstats',{}); start=st['next_group']; print('resuming from group', start)
else:
    coupled={}; cstats={}; start=0

# ---- authors' EXACT SPIN.py logic (verbatim) ----
def get_set_difference_mask(p, q, W_metric1, W_metric2):
    top_p = int(p * W_metric1.shape[1] * W_metric1.shape[0])
    top_q = int(q * W_metric2.shape[1] * W_metric2.shape[0])
    top_p_indices = torch.topk(W_metric1.flatten(), top_p, largest=True)[1]
    top_q_indices = torch.topk(W_metric2.flatten(), top_q, largest=True)[1]
    unique_p = torch.unique(top_p_indices)
    unique_q = torch.unique(top_q_indices)
    mask_only_metric1 = ~torch.isin(unique_p, unique_q)
    mask_only_metric2 = ~torch.isin(unique_q, unique_p)
    mask_intersection = ~(torch.ones_like(unique_p).bool() & mask_only_metric1 & mask_only_metric2)
    return mask_only_metric1, mask_only_metric2, mask_intersection, unique_q

def coupled_indices_for_matrix(W_general, W_fair, W_priv):
    _, mask_only_trust1, _, unique_trust1_indices = get_set_difference_mask(P, Q, W_general, W_fair)
    _, mask_only_trust2, _, unique_trust2_indices = get_set_difference_mask(P, Q, W_general, W_priv)
    mask_trust_intersection = mask_only_trust1 & mask_only_trust2
    filtered_indices = unique_trust1_indices[mask_trust_intersection]
    return filtered_indices

def coupling_set_sizes(W_general, W_fair, W_priv):
    # true SET accounting (independent of the authors' positional mask), for the coupling assumption
    def tks(W, ratio):
        k=int(ratio*W.shape[0]*W.shape[1])
        if k<1: return set()
        return set(torch.unique(torch.topk(W.flatten(), k, largest=True)[1]).tolist())
    Sg=tks(W_general,P); Sf=tks(W_fair,Q); Sp=tks(W_priv,Q)
    fp=Sf & Sp
    return {'ng':len(Sg),'nf':len(Sf),'np':len(Sp),'n_fp':len(fp),'n_fp_notg':len(fp-Sg)}

@torch.enable_grad()
def score_group(gnames, batches):
    for n,p in model.named_parameters(): p.requires_grad_(n in gnames)
    acc={n:torch.zeros_like(model.get_parameter(n)) for n in gnames}
    for inp,tar in batches:
        model.zero_grad(set_to_none=True); model(input_ids=inp,labels=tar).loss.backward()
        for n in gnames:
            gr=model.get_parameter(n).grad
            if gr is not None: acc[n]+=gr.detach().abs()
    imp={n:(model.get_parameter(n).detach().abs()*acc[n]).cpu() for n in gnames}  # to CPU, free GPU
    del acc; gc.collect(); torch.cuda.empty_cache()
    return imp

model.gradient_checkpointing_enable(); model.config.use_cache=False
for gi in range(start, len(GROUPS)):
    grp=GROUPS[gi]; gnames=[n for n in names if any(f'.layers.{L}.' in n for L in grp)]
    t0=time.time()
    imp_f=score_group(gnames,BATCHES['f']); imp_p=score_group(gnames,BATCHES['p']); imp_g=score_group(gnames,BATCHES['g'])
    for n in gnames:
        fi=coupled_indices_for_matrix(imp_g[n], imp_f[n], imp_p[n])   # authors' exact selection
        ss=coupling_set_sizes(imp_g[n], imp_f[n], imp_p[n]); ss['code']=int(len(fi)); cstats[n]=ss
        if len(fi): coupled[n]=[int(x) for x in fi.tolist()]
    del imp_f,imp_p,imp_g; gc.collect(); torch.cuda.empty_cache()
    torch.save({'coupled':coupled,'cstats':cstats,'next_group':gi+1}, CK)
    tot=sum(len(v) for v in coupled.values())
    print(f'  group {gi+1}/{len(GROUPS)} done ({time.time()-t0:.0f}s) | coupled so far {tot}'); memline()
model.gradient_checkpointing_disable()
for p in model.parameters(): p.requires_grad_(False)

meta={'model':MODEL_ID,'variant':'author-sample','sampling':'hf_datasets.shuffle(seed).select',
      'nsamples':NSAMPLES,'target':TARGET,'p':P,'q':Q,
      'coupled_count':sum(len(v) for v in coupled.values()), 'coupled':coupled}
json.dump(meta, open('/kaggle/working/coupled_indices_authorsample.json','w'))
print('\nSTEP 1 (author-sample) DONE -> coupled_indices_authorsample.json | coupled weights:', meta['coupled_count'])
print('A/B: faithful (python-shuffle) gave 1149. Delta here =', meta['coupled_count']-1149)

# ---- coupling-assumption set sizes + coupled-set structure (thesis-required reporting) ----
import re as _re
def _layer(n):
    m=_re.search(r'layers\.(\d+)\.', n); return int(m.group(1)) if m else -1
def _proj(n):
    m=_re.search(r'mlp\.(\w+?)_proj', n); return m.group(1) if m else '?'
agg={k:int(sum(cs.get(k,0) for cs in cstats.values())) for k in ['ng','nf','np','n_fp','n_fp_notg','code']}
mlp_params=int(sum(model.get_parameter(n).numel() for n in names))
per_layer={}; per_proj={}
for n,idxs in coupled.items():
    per_layer[_layer(n)]=per_layer.get(_layer(n),0)+len(idxs)
    per_proj[_proj(n)]=per_proj.get(_proj(n),0)+len(idxs)
cstats_out={'model':MODEL_ID,'p':P,'q':Q,'nsamples':NSAMPLES,'aggregate':agg,
    'coupled_total':int(sum(len(v) for v in coupled.values())),'mlp_params':mlp_params,
    'intervention_sparsity':(agg['code']/mlp_params if mlp_params else 0.0),
    'coupled_per_layer':{str(k):per_layer[k] for k in sorted(per_layer)},
    'coupled_per_proj':per_proj,'per_matrix':cstats}
json.dump(cstats_out, open('/kaggle/working/coupling_stats_authorsample.json','w'), indent=1)
print('\nCOUPLING STATS -> coupling_stats_authorsample.json')
print(f"  |N_f n N_p| summed over matrices = {agg['n_fp']}  | clean (N_f n N_p)\\N_g = {agg['n_fp_notg']}  | code-positional = {agg['code']}")
print(f"  intervention sparsity = {cstats_out['intervention_sparsity']:.2e}  ({agg['code']} / {mlp_params} MLP params)")
print('  coupled per projection:', per_proj)
